# LSMC Baseline and Finite-Difference Delta

This notebook establishes the finite-difference delta baseline for the project. It sits between the pricing replication notebook and the pathwise notebook: first we validate the pricing engine, then we examine the simplest Greek estimator built on top of it.


## Objective

The goal here is to make the bump-and-revalue baseline explicit. We use common random numbers inside LSMC, compare the resulting delta estimates to independent benchmarks, and show why bump-size sensitivity is an important limitation of the method.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent
src_root = project_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

figure_dir = project_root / "assets" / "figures"
figure_dir.mkdir(parents=True, exist_ok=True)
plt.style.use("seaborn-v0_8-whitegrid")

from lsmc_greeks.benchmarks.binomial import american_put_delta_binomial
from lsmc_greeks.benchmarks.finite_difference import american_put_delta_finite_difference
from lsmc_greeks.greeks.finite_diff import estimate_delta_fd
from lsmc_greeks.pricer import LSMCConfig, lsm_american_put

def save_figure(fig, filename):
    path = figure_dir / filename
    fig.savefig(path, dpi=220, bbox_inches="tight")
    return path


## Base Case and Benchmarks

We use the at-the-money American put as the main test case and validate the LSMC finite-difference delta against both the binomial and finite-difference PDE benchmarks.


In [ ]:
spot = 40.0
strike = 40.0
rate = 0.06
sigma = 0.20
maturity = 1.0

base_config = LSMCConfig(n_paths=40_000, n_steps_per_year=50, basis_degree=2, seed=42)
price_result = lsm_american_put(spot, strike, rate, sigma, maturity, config=base_config)
fd_lsmc = estimate_delta_fd(spot, strike, rate, sigma, maturity, config=base_config, bump=0.5, seed=42)
benchmark_binomial = american_put_delta_binomial(spot, strike, rate, sigma, maturity, n_steps=1000, bump=0.5)
benchmark_pde = american_put_delta_finite_difference(spot, strike, rate, sigma, maturity, n_space_steps=200, n_time_steps_per_year=2000, bump=0.5)


In [ ]:
baseline_table = pd.DataFrame([
    {"quantity": "LSMC American put price", "value": price_result.american_price},
    {"quantity": "LSMC finite-difference delta", "value": fd_lsmc["estimate"]},
    {"quantity": "Binomial delta benchmark", "value": benchmark_binomial},
    {"quantity": "Finite-difference PDE benchmark", "value": benchmark_pde},
    {"quantity": "Abs. error vs binomial", "value": abs(fd_lsmc["estimate"] - benchmark_binomial)},
    {"quantity": "Abs. error vs PDE", "value": abs(fd_lsmc["estimate"] - benchmark_pde)},
    {"quantity": "Runtime (sec)", "value": fd_lsmc["runtime_sec"]},
]).round(6)
baseline_table


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

axes[0].bar(["LSMC FD", "Binomial", "PDE FD"], [fd_lsmc["estimate"], benchmark_binomial, benchmark_pde], color=["#4c78a8", "#72b7b2", "#54a24b"])
axes[0].set_title("Baseline delta comparison")
axes[0].set_ylabel("Delta")

axes[1].bar(["vs binomial", "vs PDE"], [abs(fd_lsmc["estimate"] - benchmark_binomial), abs(fd_lsmc["estimate"] - benchmark_pde)], color=["#4c78a8", "#54a24b"])
axes[1].set_title("Baseline absolute error")
axes[1].set_ylabel("Absolute error")

plt.tight_layout()
save_figure(fig, "finite_difference_delta_baseline.png")
plt.show()


## Spot Sweep

We next compare the LSMC finite-difference delta to the benchmark deltas across a small moneyness grid. This checks whether the baseline method captures the right shape, not just one point.


In [ ]:
spot_rows = []
for spot_value in [36.0, 38.0, 40.0, 42.0, 44.0]:
    fd_est = estimate_delta_fd(spot_value, strike, rate, sigma, maturity, config=base_config, bump=0.5, seed=42)
    tree_est = american_put_delta_binomial(spot_value, strike, rate, sigma, maturity, n_steps=1000, bump=0.5)
    pde_est = american_put_delta_finite_difference(spot_value, strike, rate, sigma, maturity, n_space_steps=200, n_time_steps_per_year=2000, bump=0.5)
    spot_rows.append({
        "spot": spot_value,
        "lsmc_fd_delta": fd_est["estimate"],
        "binomial_delta": tree_est,
        "pde_fd_delta": pde_est,
        "abs_error_vs_binomial": abs(fd_est["estimate"] - tree_est),
    })

spot_df = pd.DataFrame(spot_rows).round(6)
spot_df


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

axes[0].plot(spot_df["spot"], spot_df["lsmc_fd_delta"], marker="^", linewidth=2, label="LSMC finite difference")
axes[0].plot(spot_df["spot"], spot_df["binomial_delta"], marker="o", linewidth=2, label="Binomial benchmark")
axes[0].plot(spot_df["spot"], spot_df["pde_fd_delta"], marker="s", linewidth=2, label="PDE benchmark")
axes[0].set_title("Delta vs spot")
axes[0].set_xlabel("Spot")
axes[0].set_ylabel("Delta")
axes[0].legend()

axes[1].plot(spot_df["spot"], spot_df["abs_error_vs_binomial"], marker="^", linewidth=2, color="#4c78a8")
axes[1].set_title("Absolute error vs spot")
axes[1].set_xlabel("Spot")
axes[1].set_ylabel("Absolute error vs binomial")

plt.tight_layout()
save_figure(fig, "finite_difference_delta_spot_sweep.png")
plt.show()


## Bump-Size Sensitivity

The main practical weakness of bump-and-revalue is that the estimate depends on a user-chosen bump. This sweep makes that dependence explicit and motivates the pathwise notebook that follows.


In [ ]:
bump_rows = []
config_bump = LSMCConfig(n_paths=20_000, n_steps_per_year=50, basis_degree=2, seed=42)
for bump in [0.10, 0.25, 0.50, 0.75, 1.00]:
    fd_est = estimate_delta_fd(spot, strike, rate, sigma, maturity, config=config_bump, bump=bump, seed=42)
    bump_rows.append({
        "bump": bump,
        "lsmc_fd_delta": fd_est["estimate"],
        "abs_error_vs_binomial": abs(fd_est["estimate"] - benchmark_binomial),
        "runtime_sec": fd_est["runtime_sec"],
    })

bump_df = pd.DataFrame(bump_rows).round(6)
bump_df


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

axes[0].plot(bump_df["bump"], bump_df["lsmc_fd_delta"], marker="^", linewidth=2, label="LSMC finite difference")
axes[0].axhline(benchmark_binomial, color="black", linestyle=":", linewidth=1.2, label="Binomial benchmark")
axes[0].axhline(benchmark_pde, color="#54a24b", linestyle="--", linewidth=1.2, label="PDE benchmark")
axes[0].set_title("Delta vs bump size")
axes[0].set_xlabel("Bump size")
axes[0].set_ylabel("Delta")
axes[0].legend()

axes[1].plot(bump_df["bump"], bump_df["abs_error_vs_binomial"], marker="^", linewidth=2, color="#4c78a8")
axes[1].set_title("Absolute error vs bump size")
axes[1].set_xlabel("Bump size")
axes[1].set_ylabel("Absolute error vs binomial")

plt.tight_layout()
save_figure(fig, "finite_difference_delta_bump_sensitivity.png")
plt.show()


## Conclusion

The finite-difference delta notebook provides the project's baseline Greek estimator. It is useful because it is straightforward and easy to benchmark, but it also shows the main limitation that motivates the next step: the estimate depends on the bump parameter. The pathwise notebook can now be presented as a direct attempt to reduce that sensitivity.
